# Introduction:

## Author: Mahmoud Tamer

If you use parts of this notebook in your own scripts, please give some credit (upvote). Thanks!


This is an attempt to use this dataset to get the best from the data by starting with understand the nature of the data and what is the best techniques to deal with missing values , drop or fill , if fill , what is the best strategy for the model 


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_JSON")


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Create the ~/.kaggle directory and place the API key
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
user_secrets = UserSecretsClient()
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(user_secrets.get_secret('KAGGLE_JSON'))

# Set file permissions for the kaggle.json file
os.system('chmod 600 ~/.kaggle/kaggle.json')


In [ ]:
!pip install kaggle --upgrade


# Import 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
%matplotlib inline

np.random.seed(42)

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import itertools

# from keras.utils.np_utils import to_categorical # convert to one-hot-encoding
from tensorflow.keras.utils import to_categorical 
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau


sns.set(style='white', context='notebook', palette='deep')

# Load Data from input folder

In [ ]:
# Load the data
train = pd.read_csv("../input/digit-recognizer/train.csv")
test = pd.read_csv("../input/digit-recognizer/test.csv")

# Create X and Y 

In [ ]:
Y_train = train["label"]

# Drop 'label' column
X_train = train.drop(labels = ["label"],axis = 1) 


In [ ]:
#  Visualize count of label
plt.figure(figsize=(10, 6))
g = sns.countplot(x=Y_train)  # Note: use x= instead of just the series
plt.title('Distribution of Digits in Y_train')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.show()
#Count of label
Y_train.value_counts()

# Null Values

In [ ]:
# Check the Train data
X_train.isnull().any().sum()

In [ ]:
# Check the test data
test.isnull().any().sum()

We can see that there is no null data in the dataset 

# Outliers 
For the digit recognizer dataset, outliers are possible but less common than in other datasets. However, they can still exist and affect model performance. Here are several methods to check for outliers:
Types of Outliers in Image Data:

    Corrupted images (all black, all white, random noise)

    Mislabeled digits

    Extreme pixel values

    Poor quality handwriting

## Method 1: Visual Inspection of Potential Outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Calculate statistics for each image to create stats_df
image_stats = []
for i in range(len(X_train)):
    if hasattr(X_train, 'iloc'):
        img = X_train.iloc[i].values
    else:
        img = X_train[i]
    
    image_stats.append({
        'mean': np.mean(img),
        'std': np.std(img),
        'max': np.max(img),
        'min': np.min(img),
        'range': np.max(img) - np.min(img)
    })

# Convert to DataFrame - this creates your stats_df
stats_df = pd.DataFrame(image_stats)
stats_df.index = X_train.index  # Use the same indices as X_train

print(f"stats_df created with shape: {stats_df.shape}")
print(stats_df.head())

In [ ]:
# Find images with extreme statistics
high_std_indices = stats_df.nlargest(30, 'std').index
low_std_indices = stats_df.nsmallest(30, 'std').index
high_mean_indices = stats_df.nlargest(30, 'mean').index
low_mean_indices = stats_df.nsmallest(30, 'mean').index

def plot_outlier_images_with_stats(indices, title, stat_type, max_cols=5):
    n_images = len(indices)
    
    # Calculate grid dimensions
    n_cols = min(max_cols, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols
    
    # Create figure with dynamic size
    fig_width = min(20, n_cols * 4)
    fig_height = n_rows * 3
    plt.figure(figsize=(fig_width, fig_height))
    
    for i, idx in enumerate(indices):
        plt.subplot(n_rows, n_cols, i+1)
        if hasattr(X_train, 'iloc'):
            img = X_train.iloc[idx].values.reshape(28, 28)
        else:
            img = X_train[idx].reshape(28, 28)
        plt.imshow(img, cmap='gray')
        
        # Get the actual statistic value
        stat_value = stats_df.loc[idx, stat_type]
        plt.title(f'Idx: {idx}\nLabel: {Y_train.iloc[idx] if hasattr(Y_train, "iloc") else Y_train[idx]}\n{stat_type}: {stat_value:.2f}', 
                 fontsize=14)
        plt.axis('off')
    
    plt.suptitle(title, fontsize=18, y=0.98)
    plt.tight_layout()
    plt.show()

# Plot with statistics displayed
plot_outlier_images_with_stats(high_std_indices, "High Standard Deviation (Noisy Images) - Top 30", 'std', max_cols=6)
plot_outlier_images_with_stats(low_std_indices, "Low Standard Deviation (Flat Images) - Bottom 20", 'std', max_cols=5)
plot_outlier_images_with_stats(high_mean_indices, "High Mean (Dark Images) - Top 20", 'mean', max_cols=5)
plot_outlier_images_with_stats(low_mean_indices, "Low Mean (Light Images) - Bottom 20", 'mean', max_cols=5)

We can see that most of them are correct except 2 idx are seems to be not cleared 11593 ,40976
let's drop them


In [ ]:
# Assuming X_train and Y_train are pandas DataFrames/Series
indices_to_drop = [11593, 40976]  # Replace with your actual indices

# Create cleaned datasets
X_train= X_train.drop(indices_to_drop)
Y_train = Y_train.drop(indices_to_drop)


print(f"Removed {len(indices_to_drop)} samples")

In [ ]:
#  reset index 
X_train = X_train.reset_index(drop=True)
Y_train = Y_train.reset_index(drop=True)

## Method 2: Check for Blank or Corrupted Images

In [ ]:
# Check for images that are mostly blank or corrupted
blank_threshold = 0.1  # If 90% of pixels are below this value

mostly_blank = []
for i in range(len(X_train)):
    if hasattr(X_train, 'iloc'):
        img = X_train.iloc[i].values
    else:
        img = X_train[i]
    
    # Check if image is mostly "blank" (low values)
    if np.mean(img) < blank_threshold:
        mostly_blank.append(i)

print(f"Mostly blank images: {len(mostly_blank)}")

if mostly_blank:
    plot_outlier_images(mostly_blank[:10], "Mostly Blank Images")

# Data Normalization 
why we normalize ?

    Faster Training - Models converge in fewer iterations

    Better Performance - Often leads to higher accuracy

    Numerical Stability - Prevents overflow/underflow issues

    Fair Feature Treatment - All features contribute equally

    Regularization Works Better - Consistent penalty across features

Important Note ! 

We have to use the same scaling for train and test data

In [ ]:
# Normalize training data
X_train = X_train / 255.0

# Normalize test data using the same scaling (important!)
test = test / 255.0

print("Training data normalized:")
print(f"Min: {X_train.min().min():.3f}, Max: {X_train.max().max():.3f}")

print("\nTest data normalized:")
print(f"Min: {test.min().min():.3f}, Max: {test.max().max():.3f}")

# Reshape

We reshape images into 3 dimensions (height, width, channels) because this is the standard format that Convolutional Neural Networks (CNNs) expect for processing image data

In [ ]:
# Reshape image in 3 dimensions (height = 28px, width = 28px , canal = 1)
X_train = X_train.values.reshape(-1,28,28,1)
test = test.values.reshape(-1,28,28,1)

# Label encoding

In [ ]:
# Encode labels to one hot vectors (ex : 2 -> [0,0,1,0,0,0,0,0,0,0])
Y_train = to_categorical(Y_train, num_classes = 10)

# Split the data into training and validation 

In [ ]:
# Split the train and the validation set for the fitting
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size = 0.2, random_state=42)

In [ ]:
# Some examples
g = plt.imshow(X_train[0][:,:,0])

# Create the model

## Create our first CNN model (Basemodel)

In [ ]:
# from tensorflow.keras import layers, models
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# import matplotlib.pyplot as plt
# import numpy as np

# # 1. Create model without augmentation
# def create_baseline_model():
#     model = models.Sequential([
#         layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
#         layers.MaxPooling2D((2,2)),
#         layers.Conv2D(64, (3,3), activation='relu'),
#         layers.MaxPooling2D((2,2)),
#         layers.Conv2D(64, (3,3), activation='relu'),
#         layers.Flatten(),
#         layers.Dense(64, activation='relu'),
#         layers.Dropout(0.5),
#         layers.Dense(10, activation='softmax')
#     ])
#     return model

# # 2. Create and compile model - USE CATEGORICAL CROSSENTROPY
# model = create_baseline_model()
# model.compile(
#     optimizer=Adam(learning_rate=0.0015),
#     loss='categorical_crossentropy',  # CHANGED THIS
#     metrics=['accuracy']
# )

# print("Model created and compiled with categorical_crossentropy!")

# # 3. Check your data shapes
# print("Data shapes:")
# print(f"X_train shape: {X_train.shape}")
# print(f"Y_train shape: {Y_train.shape}")
# print(f"X_val shape: {X_val.shape}") 
# print(f"Y_val shape: {Y_val.shape}")

# # 4. Set up callbacks
# reduce_lr = ReduceLROnPlateau(
#     monitor='val_loss',
#     factor=0.5,
#     patience=3,
#     min_lr=1e-7,
#     verbose=1
# )

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=10,
#     restore_best_weights=True,
#     verbose=1
# )

# # 5. Train the model
# print("\n" + "="*60)
# print("TRAINING BASELINE MODEL (NO AUGMENTATION)")
# print("="*60)

# history = model.fit(
#     X_train, Y_train,  # Use X_train directly since it's already reshaped
#     batch_size=24,
#     epochs=150,
#     validation_data=(X_val, Y_val),
#     callbacks=[reduce_lr, early_stop],
#     verbose=1
# )

# # 6. Evaluate and plot results
# val_loss, val_accuracy = model.evaluate(X_val, Y_val, verbose=0)

# print("\n" + "="*100)
# print("BASELINE PERFORMANCE (NO AUGMENTATION)")
# print("="*100)
# print(f"Validation Accuracy: {val_accuracy:.4f}")
# print(f"Validation Loss: {val_loss:.4f}")

# # Plot results
# plt.figure(figsize=(15, 5))

# # Accuracy plot
# plt.subplot(1, 2, 1)
# plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
# plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
# plt.title('Model Accuracy - No Augmentation', fontsize=14)
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.grid(True, alpha=0.3)

# # Loss plot
# plt.subplot(1, 2, 2)
# plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
# plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
# plt.title('Model Loss - No Augmentation', fontsize=14)
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

## Model with data Augmentaion

using data augmentaion can improve the accuracy of the model but when you use it be very careful because aggressive augmentaion may affect your model in negative way.

In [ ]:
# from tensorflow.keras import layers, models
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# from sklearn.utils import shuffle

# # 1. Data Augmentation Function - 1000 images per digit
# def augment_images_per_digit(X_train, Y_train, images_per_digit=1000):
#     """
#     Augment images to get exactly 1000 images for each digit (0-9)
#     """
#     augmented_images = []
#     augmented_labels = []
    
#     # Convert one-hot encoded labels back to categorical
#     Y_train_labels = np.argmax(Y_train, axis=1)
    
#     print("Starting augmentation: Creating 1000 images for each digit (0-9)")
#     print("="*60)
    
#     for digit in range(10):
#         print(f"Processing digit {digit}...")
        
#         # Get all original images for this digit
#         digit_indices = np.where(Y_train_labels == digit)[0]
#         digit_images = X_train[digit_indices]
        
#         if len(digit_images) == 0:
#             print(f"Warning: No images found for digit {digit}")
#             continue
            
#         # Calculate how many augmented images we need for this digit
#         needed_images = images_per_digit - len(digit_images)
        
#         if needed_images <= 0:
#             # If we already have enough images, just use the original ones
#             print(f"Digit {digit}: Already has {len(digit_images)} images (enough)")
#             augmented_images.extend(digit_images)
#             augmented_labels.extend([digit] * len(digit_images))
#             continue
        
#         print(f"Digit {digit}: Original images: {len(digit_images)}, Need to generate: {needed_images}")
        
#         # Create ImageDataGenerator for augmentation
#         datagen = ImageDataGenerator(
#             brightness_range=[0.3, 1.7],
#             rotation_range=10,
#             width_shift_range=0.1,
#             height_shift_range=0.1,
#             zoom_range=0.1
#         )
        
#         # Generate augmented images for this digit
#         aug_count = 0
#         image_index = 0
        
#         while aug_count < needed_images:
#             # Cycle through original images of this digit
#             original_img = digit_images[image_index % len(digit_images)].reshape(1, 28, 28, 1)
            
#             # Generate one augmented image
#             for batch in datagen.flow(original_img, batch_size=1):
#                 augmented_img = batch[0]
                
#                 # Apply different effects randomly
#                 effect_type = np.random.choice(['dark', 'bright', 'faded'])
                
#                 if effect_type == 'dark':
#                     # Darker effect
#                     dark_factor = np.random.uniform(0.2, 0.6)
#                     augmented_img = augmented_img * dark_factor
#                 elif effect_type == 'bright':
#                     # Brighter effect
#                     bright_factor = np.random.uniform(1.5, 2.5)
#                     augmented_img = np.clip(augmented_img * bright_factor, 0, 1)
#                 else:
#                     # Faded effect (reduce contrast)
#                     faded_factor = np.random.uniform(0.1, 0.3)
#                     mean_val = np.mean(augmented_img)
#                     augmented_img = augmented_img * (1 - faded_factor) + mean_val * faded_factor
                
#                 augmented_images.append(augmented_img)
#                 augmented_labels.append(digit)
#                 aug_count += 1
                
#                 # Move to next original image
#                 image_index += 1
#                 break
            
#             # Break if we've generated enough
#             if aug_count >= needed_images:
#                 break
        
#         # Add original images for this digit
#         augmented_images.extend(digit_images)
#         augmented_labels.extend([digit] * len(digit_images))
        
#         print(f"Digit {digit}: Total after augmentation: {len(digit_images) + needed_images} images")
    
#     # Convert to numpy arrays
#     augmented_images = np.array(augmented_images).reshape(-1, 28, 28, 1)
#     augmented_labels = np.array(augmented_labels)
    
#     # Convert labels back to one-hot encoding
#     augmented_labels_onehot = np.eye(10)[augmented_labels]
    
#     print("\n" + "="*60)
#     print("AUGMENTATION COMPLETED!")
#     print("="*60)
    
#     # Print distribution
#     unique, counts = np.unique(augmented_labels, return_counts=True)
#     for digit, count in zip(unique, counts):
#         print(f"Digit {digit}: {count} images")
    
#     print(f"\nTotal images after augmentation: {len(augmented_images)}")
    
#     return augmented_images, augmented_labels_onehot

# # 2. Create augmented dataset with exactly 1000 images per digit
# print("Creating augmented dataset with 1000 images per digit...")
# X_train_augmented, Y_train_augmented = augment_images_per_digit(X_train, Y_train, images_per_digit=1000)

# # Shuffle the augmented dataset
# X_train_augmented, Y_train_augmented = shuffle(X_train_augmented, Y_train_augmented, random_state=42)

# print(f"\nFinal augmented dataset shape:")
# print(f"X_train_augmented: {X_train_augmented.shape}")
# print(f"Y_train_augmented: {Y_train_augmented.shape}")

# # 3. Create DEEPER model with advanced architecture
# def create_deep_model():
#     model = models.Sequential([
#         # First Conv Block
#         layers.Conv2D(64, (5,5), activation='relu', padding='same', input_shape=(28,28,1)),
#         layers.BatchNormalization(),
#         layers.Conv2D(64, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.25),
        
#         # Second Conv Block
#         layers.Conv2D(128, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.Conv2D(128, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.25),
        
#         # Third Conv Block
#         layers.Conv2D(256, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.Conv2D(256, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.Conv2D(256, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.3),
        
#         # Fourth Conv Block
#         layers.Conv2D(512, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.Conv2D(512, (3,3), activation='relu', padding='same'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.4),
        
#         # Global Average Pooling
#         layers.GlobalAveragePooling2D(),
        
#         # Dense Layers
#         layers.Dense(512, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.5),
        
#         layers.Dense(256, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.5),
        
#         layers.Dense(128, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.4),
        
#         layers.Dense(10, activation='softmax')
#     ])
#     return model

# # 4. Create and compile model
# model = create_deep_model()
# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# print("\nDEEPER MODEL ARCHITECTURE:")
# print("="*50)
# model.summary()

# # 5. Enhanced callbacks
# reduce_lr = ReduceLROnPlateau(
#     monitor='val_accuracy',
#     factor=0.5,
#     patience=5,
#     min_lr=1e-7,
#     mode='max',
#     verbose=1
# )

# early_stop = EarlyStopping(
#     monitor='val_accuracy',
#     patience=15,
#     restore_best_weights=True,
#     mode='max',
#     verbose=1
# )

# # 6. Train the model with augmented data
# print("\n" + "="*60)
# print("TRAINING DEEPER MODEL WITH BALANCED AUGMENTED DATA")
# print("="*60)

# history = model.fit(
#     X_train_augmented, Y_train_augmented,  # Using only augmented data (1000 per digit)
#     batch_size=32,
#     epochs=200,
#     validation_data=(X_val, Y_val),
#     callbacks=[reduce_lr, early_stop],
#     verbose=1
# )

# # 7. Evaluate on validation set
# val_loss, val_accuracy = model.evaluate(X_val, Y_val, verbose=0)

# print("\n" + "="*100)
# print("MODEL PERFORMANCE WITH BALANCED AUGMENTATION")
# print("="*100)
# print(f"Validation Accuracy: {val_accuracy:.4f}")
# print(f"Validation Loss: {val_loss:.4f}")

# # 8. Make predictions on test data
# print("\n" + "="*60)
# print("MAKING PREDICTIONS ON TEST DATA")
# print("="*60)

# # Ensure test data is properly shaped
# if len(test.shape) == 3:
#     test_processed = test.reshape(-1, 28, 28, 1)
# else:
#     test_processed = test

# print(f"Test data shape: {test_processed.shape}")

# # Make predictions
# test_predictions = model.predict(test_processed)
# test_pred_classes = np.argmax(test_predictions, axis=1)

# print(f"Predictions shape: {test_pred_classes.shape}")
# print(f"Sample predictions: {test_pred_classes[:10]}")

# # 9. Create submission file
# submission = pd.DataFrame({
#     'ImageId': range(1, len(test_pred_classes) + 1),
#     'Label': test_pred_classes
# })

# submission_file = 'submission.csv'
# submission.to_csv(submission_file, index=False)

# print("\n" + "="*60)
# print("SUBMISSION FILE CREATED!")
# print("="*60)
# print(f"File saved as: {submission_file}")
# print(f"Submission shape: {submission.shape}")
# print(f"Label distribution in submission:")
# print(submission['Label'].value_counts().sort_index())

# # 10. Plot results
# plt.figure(figsize=(15, 5))

# # Accuracy plot
# plt.subplot(1, 2, 1)
# plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
# plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
# plt.title('Model Accuracy - Balanced Augmentation', fontsize=14, fontweight='bold')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.grid(True, alpha=0.3)

# # Loss plot
# plt.subplot(1, 2, 2)
# plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
# plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
# plt.title('Model Loss - Balanced Augmentation', fontsize=14, fontweight='bold')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# # 11. Display final metrics
# final_epoch = len(history.history['accuracy'])
# final_train_acc = history.history['accuracy'][-1]
# final_val_acc = history.history['val_accuracy'][-1]

# print(f"\n📊 FINAL TRAINING METRICS:")
# print(f"   • Final Epoch: {final_epoch}")
# print(f"   • Final Training Accuracy: {final_train_acc:.4f}")
# print(f"   • Final Validation Accuracy: {final_val_acc:.4f}")
# print(f"   • Accuracy Gap: {abs(final_train_acc - final_val_acc):.4f}")

# # 12. Show sample of test predictions
# print(f"\n🔍 SAMPLE TEST PREDICTIONS:")
# sample_indices = np.random.choice(len(test_pred_classes), 10, replace=False)
# for i, idx in enumerate(sample_indices):
#     print(f"   Sample {i+1}: ImageId {idx+1} → Predicted Label {test_pred_classes[idx]}")

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import tensorflow as tf

# 1. Data Preprocessing Function
def preprocess_data(X):
    """Normalize pixel values to [0, 1] range"""
    X = X.astype('float32')
    X /= 255.0
    return X

# Apply preprocessing to all datasets
print("Preprocessing data...")
X_train = preprocess_data(X_train)
X_val = preprocess_data(X_val)
test = preprocess_data(test)

print(f"Data shapes after preprocessing:")
print(f"X_train: {X_train.shape}, Y_train: {Y_train.shape}")
print(f"X_val: {X_val.shape}, Y_val: {Y_val.shape}")
print(f"test: {test.shape}")

# 2. Data Augmentation Function - 400 images per digit added to original
def augment_images_per_digit(X_train, Y_train, additional_images_per_digit=400):
    """
    Add 400 augmented images for each digit (0-9) to the original dataset
    """
    augmented_images = []
    augmented_labels = []
    
    # Convert one-hot encoded labels back to categorical
    Y_train_labels = np.argmax(Y_train, axis=1)
    
    print("Starting augmentation: Adding 400 images for each digit (0-9) to original data")
    print("="*70)
    
    # Print original distribution
    original_counts = np.bincount(Y_train_labels, minlength=10)
    print("Original training set distribution:")
    for digit, count in enumerate(original_counts):
        print(f"  Digit {digit}: {count} images")
    
    for digit in range(10):
        print(f"\nProcessing digit {digit}...")
        
        # Get all original images for this digit
        digit_indices = np.where(Y_train_labels == digit)[0]
        digit_images = X_train[digit_indices]
        original_count = len(digit_images)
        
        if len(digit_images) == 0:
            print(f"Warning: No images found for digit {digit}")
            continue
        
        print(f"Digit {digit}: {original_count} original images + {additional_images_per_digit} augmented")
        
        # Create ImageDataGenerator for augmentation with MILD parameters
        datagen = ImageDataGenerator(
            brightness_range=[0.8, 1.2],      # Mild brightness changes
          
            width_shift_range=0.05,           # Reduced shifting
            height_shift_range=0.05,
            zoom_range=0.05,                  # Reduced zoom
            fill_mode='nearest'
        )
        
        # Generate augmented images for this digit
        aug_count = 0
        image_index = 0
        
        while aug_count < additional_images_per_digit:
            # Cycle through original images of this digit
            original_img = digit_images[image_index % len(digit_images)].reshape(1, 28, 28, 1)
            
            # Generate one augmented image
            for batch in datagen.flow(original_img, batch_size=1):
                augmented_img = batch[0]
                
                # Apply different effects with MILD parameters
                effect_type = np.random.choice(['dark', 'bright', 'faded'])
                
                if effect_type == 'dark':
                    # Mild dark effect
                    dark_factor = np.random.uniform(0.94, 0.98)    # Less extreme
                    augmented_img = augmented_img * dark_factor
                elif effect_type == 'bright':
                    # Mild bright effect
                    bright_factor = np.random.uniform(1.05, 1.08)  # Less extreme
                    augmented_img = np.clip(augmented_img * bright_factor, 0, 1)
                else:
                    # Mild faded effect
                    faded_factor = np.random.uniform(0.05, 0.08) # Less extreme
                    mean_val = np.mean(augmented_img)
                    augmented_img = augmented_img * (1 - faded_factor) + mean_val * faded_factor
                
                augmented_images.append(augmented_img)
                augmented_labels.append(digit)
                aug_count += 1
                
                # Move to next original image
                image_index += 1
                break
            
            # Break if we've generated enough
            if aug_count >= additional_images_per_digit:
                break
        
        print(f"Digit {digit}: Generated {aug_count} augmented images")
    
    # Convert to numpy arrays
    augmented_images = np.array(augmented_images).reshape(-1, 28, 28, 1)
    augmented_labels = np.array(augmented_labels)
    
    # Convert labels back to one-hot encoding
    augmented_labels_onehot = np.eye(10)[augmented_labels]
    
    print("\n" + "="*70)
    print("AUGMENTATION COMPLETED!")
    print("="*70)
    
    # Combine original and augmented data
    X_combined = np.concatenate([X_train, augmented_images])
    Y_combined = np.concatenate([Y_train, augmented_labels_onehot])
    
    # Print final distribution
    Y_combined_labels = np.argmax(Y_combined, axis=1)
    final_counts = np.bincount(Y_combined_labels, minlength=10)
    
    print("FINAL COMBINED DATASET DISTRIBUTION:")
    total_original = len(X_train)
    total_augmented = len(augmented_images)
    for digit in range(10):
        original_count = original_counts[digit]
        final_count = final_counts[digit]
        augmented_count = final_count - original_count
        print(f"Digit {digit}: {final_count} total ({original_count} original + {augmented_count} augmented)")
    
    print(f"\nDataset Summary:")
    print(f"Original training images: {total_original}")
    print(f"Augmented images added: {total_augmented}")
    print(f"Total combined images: {len(X_combined)}")
    
    return X_combined, Y_combined

# 3. Create combined dataset (original + 400 augmented per digit)
print("\nCreating combined dataset: Original + 400 augmented images per digit...")
X_combined, Y_combined = augment_images_per_digit(X_train, Y_train, additional_images_per_digit=400)

# Shuffle the combined dataset
X_combined, Y_combined = shuffle(X_combined, Y_combined, random_state=42)

print(f"\nFinal combined dataset shape:")
print(f"X_combined: {X_combined.shape}")
print(f"Y_combined: {Y_combined.shape}")

# 4. Create IMPROVED model with optimized architecture
def create_improved_model():
    model = models.Sequential([
        # First Conv Block - Reduced regularization
        layers.Conv2D(64, (5,5), activation='relu', padding='same', input_shape=(28,28,1)),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.2),  # Reduced from 0.25
        
        # Second Conv Block
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        
        # Third Conv Block
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.3),
        
        # Fourth Conv Block - Simplified
        layers.Conv2D(512, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.35),  # Reduced from 0.4
        
        # Global Average Pooling
        layers.GlobalAveragePooling2D(),
        
        # Dense Layers - Reduced dropout
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),  # Reduced from 0.5
        
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),  # Reduced from 0.5
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),  # Reduced from 0.4
        
        layers.Dense(10, activation='softmax')
    ])
    return model

# 5. Create and compile model
model = create_improved_model()
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nIMPROVED MODEL ARCHITECTURE:")
print("="*50)
model.summary()

# 6. Enhanced callbacks
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=8,  # Increased patience
    min_lr=1e-6, # Higher minimum learning rate
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,  # Increased patience
    restore_best_weights=True,
    mode='max',
    verbose=1
)

# Model checkpoint to save best model
checkpoint = ModelCheckpoint(
    'best_digit_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# 7. Train the model on combined data
print("\n" + "="*60)
print("TRAINING IMPROVED MODEL WITH COMBINED DATA")
print("="*60)
print("Using: Original data + 400 augmented images per digit")
print("="*60)

history = model.fit(
    X_combined, Y_combined,
    batch_size=64,  # Increased batch size for stability
    epochs=200,
    validation_data=(X_val, Y_val),
    callbacks=[reduce_lr, early_stop, checkpoint],
    verbose=1
)

# 8. Comprehensive Evaluation Function
def comprehensive_evaluation(model, X_val, Y_val):
    """Provide detailed model evaluation"""
    # Get predictions
    val_predictions = model.predict(X_val, verbose=0)
    val_pred_classes = np.argmax(val_predictions, axis=1)
    val_true_classes = np.argmax(Y_val, axis=1)
    
    # Calculate accuracy
    val_accuracy = np.mean(val_pred_classes == val_true_classes)
    
    # Classification report
    print("\n" + "="*60)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(val_true_classes, val_pred_classes, digits=4))
    
    # Confusion matrix
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(val_true_classes, val_pred_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # Per-class accuracy
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    print("\nPer-class Accuracy:")
    for i, acc in enumerate(class_accuracy):
        print(f"  Digit {i}: {acc:.4f}")
    
    return val_accuracy

# 9. Evaluate on validation set
print("\n" + "="*100)
print("COMPREHENSIVE MODEL EVALUATION")
print("="*100)

val_accuracy = comprehensive_evaluation(model, X_val, Y_val)
val_loss, val_accuracy_keras = model.evaluate(X_val, Y_val, verbose=0)

print(f"\nOverall Validation Accuracy: {val_accuracy_keras:.4f}")
print(f"Overall Validation Loss: {val_loss:.4f}")

# 10. Make predictions on test data
print("\n" + "="*60)
print("MAKING PREDICTIONS ON TEST DATA")
print("="*60)

# Ensure test data is properly shaped
if len(test.shape) == 3:
    test_processed = test.reshape(-1, 28, 28, 1)
else:
    test_processed = test

print(f"Test data shape: {test_processed.shape}")

# Make predictions
test_predictions = model.predict(test_processed, verbose=0)
test_pred_classes = np.argmax(test_predictions, axis=1)
test_confidences = np.max(test_predictions, axis=1)

print(f"Predictions shape: {test_pred_classes.shape}")
print(f"Sample predictions with confidence:")
for i in range(10):
    print(f"  Image {i+1}: Predicted {test_pred_classes[i]} (confidence: {test_confidences[i]:.3f})")

# 11. Create submission file
submission = pd.DataFrame({
    'ImageId': range(1, len(test_pred_classes) + 1),
    'Label': test_pred_classes
})

submission_file = 'submission.csv'
submission.to_csv(submission_file, index=False)

print("\n" + "="*60)
print("SUBMISSION FILE CREATED!")
print("="*60)
print(f"File saved as: {submission_file}")
print(f"Submission shape: {submission.shape}")
print(f"Label distribution in submission:")
label_dist = submission['Label'].value_counts().sort_index()
for label, count in label_dist.items():
    print(f"  Digit {label}: {count} images")

# 12. Enhanced Plotting
plt.figure(figsize=(18, 6))

# Accuracy plot
plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Model Accuracy - 400 Augmented/Digit', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Loss plot
plt.subplot(1, 3, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Model Loss - 400 Augmented/Digit', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Learning rate plot
plt.subplot(1, 3, 3)
if 'lr' in history.history:
    plt.plot(history.history['lr'], label='Learning Rate', linewidth=2, color='red')
    plt.title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.legend()
    plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.tight_layout()
plt.show()

# 13. Display final metrics
final_epoch = len(history.history['accuracy'])
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"\n📊 FINAL TRAINING METRICS:")
print(f"   • Final Epoch: {final_epoch}")
print(f"   • Final Training Accuracy: {final_train_acc:.4f}")
print(f"   • Final Validation Accuracy: {final_val_acc:.4f}")
print(f"   • Accuracy Gap: {abs(final_train_acc - final_val_acc):.4f}")

# 14. Display sample test images with predictions
print(f"\n🔍 SAMPLE TEST PREDICTIONS:")
plt.figure(figsize=(15, 8))
sample_indices = np.random.choice(len(test_pred_classes), 15, replace=False)

for i, idx in enumerate(sample_indices):
    plt.subplot(3, 5, i+1)
    
    # Display test image
    if test_processed[idx].shape == (28, 28, 1):
        plt.imshow(test_processed[idx].reshape(28, 28), cmap='gray')
    else:
        plt.imshow(test_processed[idx], cmap='gray')
    
    plt.title(f'Img {idx+1}: Pred {test_pred_classes[idx]}\nConf: {test_confidences[idx]:.3f}', 
              fontsize=9, pad=2)
    plt.axis('off')

plt.suptitle('Sample Test Images with Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# 15. Training Summary
print(f"\n🎯 TRAINING SUMMARY:")
print(f"   • Training data: {len(X_combined)} images total")
print(f"   • Validation data: {len(X_val)} images")
print(f"   • Test data: {len(test_processed)} images")
print(f"   • Augmentation: 400 images per digit added to original data")
print(f"   • Best model saved as: best_digit_model.h5")
print(f"   • Submission file: {submission_file}")

print(f"\n✅ TRAINING COMPLETED SUCCESSFULLY!")